In [ ]:
import pandas as pd

In [ ]:
credits=pd.read_csv("tmdb_5000_credits.csv")
movies=pd.read_csv("tmdb_5000_movies.csv")

In [ ]:
df=movies.merge(credits,on='title')

In [ ]:
df.shape

In [ ]:
#to see a specific row/column in dataframe 
df.iloc[[2]][['original_title']]

In [ ]:
df.sample(1)

In [ ]:
# column gotta help for recommendation system 
# 'genres','id','keywords','title','overview','cast','crew'

In [ ]:
df=df[['genres','id','keywords','title','overview','cast','crew']]

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.iloc[0].genres

In [ ]:
import ast
def convert(obj):
    l=[]
    for i in ast.literal_eval(obj):
        l.append(i['name'])
    return l

In [ ]:
df['genres']=df['genres'].apply(convert)

In [ ]:
df['keywords']=df['keywords'].apply(convert)

In [ ]:
df

In [ ]:
import ast
def convert_new(obj):
    l=[]
    counter=0
    for i in ast.literal_eval(obj):
        if counter !=3:
            l.append(i['name'])
            counter +=1
        else:
            break
    return l

In [ ]:
df['cast']=df['cast'].apply(convert_new)

In [ ]:
df

In [ ]:
df['crew'][0]

In [ ]:
def fetch_director(obj):
    l=[]
    for i in ast.literal_eval(obj):
        if i['job']== "Director":
            l.append(i['name'])
            break
    return l

In [ ]:
df['crew']=df['crew'].apply(fetch_director)

In [ ]:
df

In [ ]:
df['overview']=df['overview'].apply(lambda x:x.split())

In [ ]:
df

In [ ]:
df['keywords']=df['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
df['genres']=df['genres'].apply(lambda x:[i.replace(" ","") for i in x])
df['cast']=df['cast'].apply(lambda x:[i.replace(" ","") for i in x])
df['crew']=df['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [ ]:
df

In [ ]:
df['tags']=df['genres']+ df['keywords'] + df['overview'] + df['cast']+ df['crew']

In [ ]:
df

In [ ]:
dff=df[['id','title','tags']]

In [ ]:
dff

In [ ]:
dff['tags']=dff['tags'].apply(lambda x:" ".join(x))

In [ ]:
dff['tags'][0]

In [ ]:
dff['tags']=dff['tags'].apply(lambda x:x.lower())

In [ ]:
# import nltk
# from nltk.stem.porter import PorterStemmer
# ps=PorterStemmer()

# def stem(text):
#     y=[]
#     for i in text.split():
#         y.append(ps.stem(i))
        
#     return " ".join(y)


In [ ]:
import numpy as np 
from sklearn.feature_extraction.text import CountVectorizer 

In [ ]:
cv=CountVectorizer(max_features=5000, stop_words='english') 

In [ ]:
vector=cv.fit_transform(dff['tags']).toarray()

In [ ]:
len(cv.get_feature_names())

In [ ]:
import nltk
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [ ]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
        
    return " ".join(y)

In [ ]:
dff['tags']=dff['tags'].apply(stem)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vector)

In [ ]:
similarity[5]

In [ ]:
def recommend(movie):
    movie_index=dff[dff['title']==movie].index[0]
    distance = similarity[movie_index]
    movie_list=sorted(list(enumerate(distance)), reverse=True, key=lambda x:x[1])[1:6]
    
    for i in movie_list:
        print(dff.iloc[i[0]].title)
    

In [ ]:
recommend('Batman Begins')

In [ ]:
# to find index
# dff[dff['title']=='Batman Begins'].index[0]

In [ ]:
# enumerated to not loose index values 
# sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x:x[1])[1:6]

In [ ]:
dff.iloc[1216].title

In [ ]:
import pickle 
pickle.dump(dff,open('movies.pkl','wb'))

In [ ]:
pickle.dump(dff.to_dict(),open('movie_dict.pkl','wb'))

In [ ]:
pickle.dump(similarity,open('similarity.pkl','wb'))